In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [3]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

# Build a Text Cleaning Pipeline
def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Cleans and preprocesses text data.
    Steps:
    - Lowercasing
    - Remove URLs
    - Remove emojis
    - Remove punctuation & special characters
    - Tokenization
    - Remove stopwords
    - Stemming or Lemmatization
    """

    # Convert to lowercase
    data = dataset.lower()

    # Remove URLs
    data = re.sub(r'http\S+|www\S+', '', data)

    # Remove emojis (basic range)
    data = re.sub(r'[^\x00-\x7F]+', '', data)

    # Remove punctuation and special characters
    data = data.translate(str.maketrans('', '', string.punctuation))
    data = re.sub(r'[^a-zA-Z\s]', '', data)

    # Tokenization
    tokens = data.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization or Stemming
    if rule == "lemmatize":
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    elif rule == "stem":
        tokens = [stemmer.stem(word) for word in tokens]

    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [5]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/AI/Week8/trum_tweet_sentiment_analysis.csv")

df.head()

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [6]:
df['clean_text'] = df['text'].apply(lambda x: text_cleaning_pipeline(x, rule="lemmatize"))

In [11]:
print(df.columns)

Index(['text', 'Sentiment', 'clean_text'], dtype='object')


In [13]:
from sklearn.model_selection import train_test_split

# features and labels
X = df['text']
y = df['Sentiment']

# Step 1: split FIRST
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Step 2: apply your preprocessing function
X_train = X_train.apply(lambda x: text_cleaning_pipeline(x, rule="lemmatize"))
X_test = X_test.apply(lambda x: text_cleaning_pipeline(x, rule="lemmatize"))

In [15]:
import re

def lower_order(text):
    return text.lower()

def remove_urls(text):
    return re.sub(r'http\S+|www\S+', '', text)

def remove_emoji(text):
    return text.encode('ascii', 'ignore').decode('ascii')

In [16]:
X_train = X_train.apply(lower_order).apply(remove_urls).apply(remove_emoji)
X_test = X_test.apply(lower_order).apply(remove_urls).apply(remove_emoji)

In [18]:
def text_cleaning_pipeline(text):
    text = lower_order(text)
    text = remove_urls(text)
    text = remove_emoji(text)
    text = removeunwanted_characters(text)
    return text

In [21]:
def text_cleaning_pipeline(text):
    text = text.lower()
    text = remove_urls(text)
    text = remove_emoji(text)
    return text

In [22]:
df["clean_text"] = df["text"].apply(text_cleaning_pipeline)

In [23]:
X = df["clean_text"]
y = df["Sentiment"]

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [26]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [27]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.97      0.97    248563
           1       0.94      0.92      0.93    121462

    accuracy                           0.95    370025
   macro avg       0.95      0.94      0.95    370025
weighted avg       0.95      0.95      0.95    370025

